# Machamba IA — Treino do classificador de pragas e doenças

<a href="https://colab.research.google.com/github/saudiapaulo/machamba-ia/blob/main/notebooks/treino_modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

Este notebook treina o modelo que vai olhar para a foto enviada pelo agricultor e dizer **que cultura é** e **que praga ou doença tem**, com um **nível de confiança**.

**O que vamos fazer:**
1. Montar o Google Drive e verificar o espaço livre
2. Obter o dataset (download directo do Mendeley **ou** zip guardado no Drive)
3. Detectar as 22 classes, limpar imagens estragadas e ver exemplos
4. Dividir em treino / validação / teste (70 / 15 / 15)
5. Preparar o *data augmentation* (só no treino)
6. Criar o modelo EfficientNet-B0 pré-treinado e tratar o desequilíbrio de classes
7. Treinar, com checkpoint no Drive para retomar se o Colab desligar
8. Avaliar no conjunto de **teste**
9. Exportar o modelo e os ficheiros para o projecto

**Antes de correr:** `Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)`.

---

### Porque usamos só a "Raw Data" (e não a versão "Augmented")

O dataset CCMT vem em duas versões: **Raw Data** (as fotos originais) e **CCMT Dataset-Augmented** (fotos já modificadas — rodadas, viradas, com outras cores — e já divididas em treino e teste).

O problema da versão *Augmented* é que **o augmentation foi feito antes da divisão**. Isso quer dizer que a mesma foto original pode aparecer, rodada, no treino e, virada, no teste. O modelo "já viu" aquela folha durante o treino e acerta no teste por **memória**, não por saber reconhecer a doença. A isto chama-se **fuga de dados** (*data leakage*): a accuracy no teste parece óptima, mas é falsa — no campo, com fotos novas, o modelo falharia muito mais.

Por isso fazemos tudo pela ordem certa: **primeiro dividimos** as fotos originais, e **só depois** aplicamos augmentation, e apenas ao conjunto de treino. Assim o teste contém só fotos que o modelo nunca viu, em nenhuma versão.

## 0. Configuração

Todos os parâmetros ficam nesta célula, para não andarmos à procura de números espalhados pelo notebook.

👉 **Se tiveres o link directo do Mendeley, cola-o em `DATASET_URL`.** Se o deixares vazio, o notebook usa o zip que está no teu Drive (ver secção 2).

In [ ]:
# ===================== Configuração (muda aqui, e só aqui) =====================

# Opção A: link directo do Mendeley (zip completo, ~9 GB). Vazio = usar a opção B.
DATASET_URL = ""

# Opção B: zip só com a "Raw Data", guardado no teu Google Drive
DRIVE_ROOT = "/content/drive/MyDrive"
DRIVE_PROJECT = f"{DRIVE_ROOT}/machamba-ia"
DRIVE_ZIP_PATH = f"{DRIVE_PROJECT}/dataset/raw_data.zip"

# Onde ficam os resultados no Drive (pouco espaço: só o essencial vai para lá)
DRIVE_MODELS_DIR = f"{DRIVE_PROJECT}/models"
DRIVE_CKPT_PATH = f"{DRIVE_MODELS_DIR}/best.pt"
FINAL_MODEL_NAME = "machamba_efficientnet_b0.pt"

# Disco local do Colab: rápido e grande, mas apagado quando a sessão termina
WORK_DIR = "/content/work"
DOWNLOAD_ZIP_PATH = "/content/ccmt_full.zip"
LOCAL_ZIP_PATH = "/content/raw_data.zip"
EXTRACT_DIR = "/content/dataset"
CACHE_DIR = "/content/cache"
BEST_LOCAL_PATH = f"{WORK_DIR}/best_local.pt"
DELETE_ZIP_AFTER_EXTRACT = True   # apaga o zip depois de extrair, para libertar disco

# Treino
SEED = 42
IMAGE_SIZE = 224        # tamanho de entrada do EfficientNet-B0
RESIZE_SIZE = 256       # lado menor da imagem antes do recorte de 224
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
PATIENCE = 4            # early stopping: pára após 4 épocas sem melhorar
MIN_EPOCHS_BETWEEN_DRIVE_SAVES = 3   # poupa escritas (e espaço) no Drive
NUM_WORKERS = 2         # o Colab gratuito tem 2 CPUs

# Limites de segurança
MIN_DRIVE_FREE_MB = 60       # abaixo disto, avisamos
MIN_LOCAL_FREE_GB = 25       # para o download de 9 GB + extracção
MIN_ZIP_BYTES = 100 * 1024**2  # um zip mais pequeno que isto está de certeza incompleto

# Normalização usada no pré-treino do ImageNet (o modelo espera estes valores)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Correcções de ortografia — só no nome apresentado, as pastas não mudam
DISPLAY_FIXES = {"grasshoper": "grasshopper"}

# Culturas esperadas e o nome em português (útil para o bot, na Fase 2)
CROP_PT = {"Cashew": "caju", "Cassava": "mandioca", "Maize": "milho", "Tomato": "tomate"}
EXPECTED_CROPS = set(CROP_PT)
EXPECTED_NUM_CLASSES = 22
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

In [ ]:
!pip install -q timm

import json
import os
import random
import shutil
import subprocess
import time
import zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import timm
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm


def set_seed(seed: int) -> None:
    """Fixa as sementes aleatórias para a divisão e o treino serem repetíveis."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
os.makedirs(WORK_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError(
        "Não há GPU! Vai a 'Ambiente de execução → Alterar tipo de ambiente de execução', "
        "escolhe 'GPU T4' e corre o notebook de novo desde o início."
    )
use_amp = True
torch.backends.cudnn.benchmark = True  # escolhe os algoritmos mais rápidos para a GPU
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | timm {timm.__version__}")

## 1. Montar o Google Drive e verificar o espaço

O Drive é **lento** para ler milhares de imagens e o teu tem **pouco espaço livre**. Por isso:
- **todo o trabalho pesado é feito em `/content`** (o disco local do Colab, com dezenas de GB);
- o Drive só recebe o essencial: um checkpoint (~16 MB) durante o treino e, no fim, o modelo e três ficheiros pequenos.

⚠️ **Atenção ao Lixo do Drive:** quando um ficheiro é apagado ou substituído no Drive, a versão antiga pode ir para o **Lixo** e **continuar a contar para a quota**. É por isso que escrevemos no Drive poucas vezes. No fim do treino, esvazia o Lixo (drive.google.com → Lixo → Esvaziar o lixo).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


def drive_free_mb() -> float:
    """Espaço livre no Google Drive, em MB (valor aproximado)."""
    return shutil.disk_usage(DRIVE_ROOT).free / 1024**2


def local_free_gb() -> float:
    """Espaço livre no disco local do Colab, em GB."""
    return shutil.disk_usage("/content").free / 1024**3


os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)

print(f"Espaço livre no Google Drive: {drive_free_mb():.0f} MB")
print(f"Espaço livre no disco do Colab: {local_free_gb():.1f} GB")

if drive_free_mb() < MIN_DRIVE_FREE_MB:
    print(
        f"\n⚠️  AVISO: o Drive tem menos de {MIN_DRIVE_FREE_MB} MB livres.\n"
        "   O checkpoint (~16 MB) e o modelo final (~16 MB) podem não caber.\n"
        "   Esvazia o Lixo do Drive ou apaga ficheiros antes de treinar."
    )
else:
    print("✅ Espaço no Drive suficiente para o checkpoint e os resultados.")

## 2. Obter o dataset

Há duas formas, tentadas **por esta ordem**:

**A) Download directo do Mendeley para `/content`** (se `DATASET_URL` estiver preenchido)
- O zip completo tem **~9 GB** e inclui "Raw Data" **e** "CCMT Dataset-Augmented".
- Extraímos **só a pasta `Raw Data/`** — poupa tempo e disco, e evita usar por engano a versão com fuga de dados.
- Verificamos se o download terminou bem: o **tamanho** tem de bater certo com o que o servidor anunciou, e o ficheiro tem de ser um **zip válido**.

**B) Zip guardado no teu Drive** (se A falhar ou o URL estiver vazio)
- `MyDrive/machamba-ia/dataset/raw_data.zip` é copiado para `/content` (uma leitura sequencial do Drive é rápida; milhares de leituras pequenas é que não) e extraído lá.

**Como obter o link directo do Mendeley:** abre https://data.mendeley.com/datasets/bwh3zbpkpv, clica com o botão direito em **"Download All"** e escolhe **"Copiar endereço do link"**. Se o botão não der um link, começa o download no browser, abre a lista de transferências (no Chrome: `chrome://downloads`), copia o link da transferência e cancela-a. Estes links podem **expirar** passado algum tempo — cola-o e corre logo a seguir.

In [ ]:
def get_remote_size(url: str) -> int | None:
    """Pergunta ao servidor o tamanho do ficheiro, sem o descarregar."""
    try:
        with requests.get(url, stream=True, allow_redirects=True, timeout=60) as response:
            response.raise_for_status()
            size = response.headers.get("Content-Length")
            return int(size) if size else None
    except Exception as error:
        print(f"   Não foi possível saber o tamanho remoto: {error}")
        return None


def download_with_wget(url: str, dest: str) -> bool:
    """Descarrega com wget, mostrando o progresso a cada 30 segundos."""
    log_path = "/content/wget.log"
    command = ["wget", "--tries=5", "--timeout=60", "--no-verbose", "-O", dest, url]
    start = last_print = time.time()
    with open(log_path, "w") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            time.sleep(5)
            if time.time() - last_print >= 30 and os.path.exists(dest):
                minutes = (time.time() - start) / 60
                print(f"   ... {os.path.getsize(dest) / 1024**3:.2f} GB descarregados ({minutes:.1f} min)")
                last_print = time.time()
    if process.returncode != 0:
        print(f"   wget terminou com erro (código {process.returncode}). Últimas linhas do registo:")
        print(open(log_path).read()[-1500:])
        return False
    return True


def validate_zip(path: str, expected_size: int | None) -> bool:
    """Confirma que o zip está completo: tamanho certo e estrutura de zip válida."""
    if not os.path.exists(path):
        print(f"   ❌ O ficheiro {path} não existe.")
        return False
    size = os.path.getsize(path)
    print(f"   Tamanho do ficheiro: {size / 1024**3:.2f} GB")
    if expected_size is not None and size != expected_size:
        print(f"   ❌ Tamanho errado: esperava {expected_size / 1024**3:.2f} GB. O download ficou incompleto.")
        return False
    if size < MIN_ZIP_BYTES:
        print("   ❌ O ficheiro é demasiado pequeno para ser o dataset (talvez seja uma página de erro).")
        return False
    try:
        # Abrir o zip lê o "índice", que está no FIM do ficheiro.
        # Se o download foi cortado, o índice não existe e isto falha.
        with zipfile.ZipFile(path) as zf:
            print(f"   ✅ Zip válido, com {len(zf.infolist())} entradas.")
        return True
    except zipfile.BadZipFile as error:
        print(f"   ❌ Não é um zip válido: {error}")
        return False


def is_raw_data_member(name: str) -> bool:
    """True se o caminho dentro do zip estiver na pasta 'Raw Data'."""
    parts = name.split("/")
    return "Raw Data" in parts and "__MACOSX" not in parts


def extract_raw_data(zip_path: str, dest: str) -> None:
    """Extrai apenas a pasta 'Raw Data/' do zip, ignorando o resto."""
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.infolist() if is_raw_data_member(m.filename)]

        if not members:
            # Alguns downloads do Mendeley trazem zips dentro do zip.
            inner = [m for m in zf.infolist()
                     if m.filename.lower().endswith(".zip") and "raw" in os.path.basename(m.filename).lower()]
            if not inner:
                raise RuntimeError("Não encontrei a pasta 'Raw Data/' dentro do zip.")
            print(f"   A 'Raw Data' está num zip interno: {inner[0].filename}")
            inner_path = zf.extract(inner[0], "/content/inner_zip")
            extract_raw_data(inner_path, dest)
            os.remove(inner_path)
            return

        damaged = []
        for member in tqdm(members, desc="A extrair Raw Data"):
            try:
                zf.extract(member, dest)
            except (zipfile.BadZipFile, OSError):
                damaged.append(member.filename)
                partial = os.path.join(dest, member.filename)
                if os.path.isfile(partial):
                    os.remove(partial)
        print(f"   ✅ Extraídas {len(members) - len(damaged)} entradas.")
        if damaged:
            print(f"   ⚠️  {len(damaged)} ficheiros estragados dentro do zip foram ignorados.")


def find_dataset_root(base: str) -> str | None:
    """Procura a pasta que contém as 4 culturas (Cashew, Cassava, Maize, Tomato)."""
    if not os.path.isdir(base):
        return None
    for dirpath, dirnames, _ in os.walk(base):
        if EXPECTED_CROPS.issubset(dirnames):
            return dirpath
    return None


def obtain_from_url(url: str) -> bool:
    """Opção A: descarrega o zip completo do Mendeley e extrai só a Raw Data."""
    print("➡️  Opção A: download directo do Mendeley")
    if local_free_gb() < MIN_LOCAL_FREE_GB:
        print(f"   ⚠️  Só há {local_free_gb():.1f} GB livres no Colab; o download pode não caber.")
    expected_size = get_remote_size(url)
    if expected_size:
        print(f"   O servidor anuncia {expected_size / 1024**3:.2f} GB.")

    if os.path.exists(DOWNLOAD_ZIP_PATH) and validate_zip(DOWNLOAD_ZIP_PATH, expected_size):
        print("   O zip já tinha sido descarregado nesta sessão; não descarrego de novo.")
    else:
        if os.path.exists(DOWNLOAD_ZIP_PATH):
            os.remove(DOWNLOAD_ZIP_PATH)
        print("   A descarregar (pode demorar 5–20 minutos)...")
        if not download_with_wget(url, DOWNLOAD_ZIP_PATH):
            return False
        if not validate_zip(DOWNLOAD_ZIP_PATH, expected_size):
            return False

    extract_raw_data(DOWNLOAD_ZIP_PATH, EXTRACT_DIR)
    if DELETE_ZIP_AFTER_EXTRACT:
        os.remove(DOWNLOAD_ZIP_PATH)
    return True


def obtain_from_drive() -> bool:
    """Opção B: copia o raw_data.zip do Drive para /content e extrai-o."""
    print("➡️  Opção B: zip guardado no Google Drive")
    if not os.path.exists(DRIVE_ZIP_PATH):
        print(f"   ❌ Não encontrei {DRIVE_ZIP_PATH}")
        return False
    drive_size = os.path.getsize(DRIVE_ZIP_PATH)
    print(f"   A copiar {drive_size / 1024**3:.2f} GB do Drive para /content...")
    start = time.time()
    shutil.copyfile(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
    print(f"   Cópia feita em {(time.time() - start) / 60:.1f} min.")
    if not validate_zip(LOCAL_ZIP_PATH, drive_size):
        return False
    extract_raw_data(LOCAL_ZIP_PATH, EXTRACT_DIR)
    if DELETE_ZIP_AFTER_EXTRACT:
        os.remove(LOCAL_ZIP_PATH)
    return True

In [ ]:
dataset_root = find_dataset_root(EXTRACT_DIR)

if dataset_root:
    print(f"O dataset já está extraído nesta sessão: {dataset_root}")
else:
    ok = False
    if DATASET_URL.strip():
        try:
            ok = obtain_from_url(DATASET_URL.strip())
        except Exception as error:
            print(f"   ❌ A opção A falhou: {error}")
        if not ok:
            print("\nA opção A não funcionou. Vou tentar a opção B.\n")
            shutil.rmtree(EXTRACT_DIR, ignore_errors=True)  # limpa uma extracção a meio
    else:
        print("DATASET_URL está vazio: uso directamente a opção B.\n")

    if not ok:
        ok = obtain_from_drive()

    if not ok:
        raise RuntimeError("Não consegui obter o dataset por nenhuma das opções. Vê as mensagens acima.")
    dataset_root = find_dataset_root(EXTRACT_DIR)

if dataset_root is None:
    raise RuntimeError(f"Extraí o zip, mas não encontrei as pastas {sorted(EXPECTED_CROPS)}.")
print(f"\n✅ Pasta do dataset: {dataset_root}")
print(f"Espaço livre no Colab: {local_free_gb():.1f} GB")

### Árvore de pastas

Confirma que a estrutura é a esperada: `Raw Data / CCMT Dataset / <cultura> / <doença>`, com 22 pastas de doença no total.

In [ ]:
def is_image_file(name: str) -> bool:
    """True se a extensão do ficheiro for de imagem."""
    return os.path.splitext(name)[1].lower() in IMAGE_EXTENSIONS


def count_files(folder: str) -> tuple[int, int]:
    """Conta (imagens, outros ficheiros) dentro de uma pasta, incluindo subpastas."""
    images = others = 0
    for _, _, files in os.walk(folder):
        for name in files:
            if is_image_file(name):
                images += 1
            else:
                others += 1
    return images, others


def list_subdirs(folder: str) -> list[str]:
    """Subpastas visíveis, por ordem alfabética."""
    return sorted(d for d in os.listdir(folder)
                  if os.path.isdir(os.path.join(folder, d)) and not d.startswith((".", "__")))


print(os.path.relpath(dataset_root, EXTRACT_DIR) + "/")
for crop in list_subdirs(dataset_root):
    crop_dir = os.path.join(dataset_root, crop)
    print(f"├── {crop}/")
    for disease in list_subdirs(crop_dir):
        images, others = count_files(os.path.join(crop_dir, disease))
        extra = f", {others} outros ficheiros" if others else ""
        print(f"│   ├── {disease}/  ({images} imagens{extra})")

## 3. Classes, limpeza e exemplos

**Detecção automática das classes.** Cada pasta `<cultura>/<doença>` é uma classe, com o nome `"Cultura - doença"` (ex.: `"Maize - fall armyworm"`). A ordem é alfabética, para o índice de cada classe ser sempre o mesmo. O erro `grasshoper` é corrigido para `grasshopper` **só no nome apresentado** — a pasta continua igual.

**Limpeza e cache.** Abrimos cada imagem uma vez para:
1. **descobrir as corrompidas** (se não abrir, fica de fora — um ficheiro estragado a meio do treino faria o treino parar);
2. **corrigir a orientação** (fotos de telemóvel guardam a rotação em separado, nos dados EXIF);
3. **reduzir** a imagem para 256 px no lado menor e guardá-la em `/content/cache`.

O passo 3 é uma optimização importante: as fotos originais são grandes, e descomprimi-las em cada época tornaria o CPU do Colab o gargalo (a GPU ficaria parada à espera). Reduzindo uma vez, cada época fica várias vezes mais rápida. O modelo trabalha com 224 px, por isso não perdemos nada de útil.

> Na Fase 2, as fotos dos agricultores vão passar pelo **mesmo** pré-processamento (orientação EXIF → RGB → lado menor 256 → recorte central 224). Por isso estes valores ficam guardados no `labels.json`.

In [ ]:
def display_name(text: str) -> str:
    """Corrige erros de ortografia conhecidos no nome apresentado."""
    for wrong, right in DISPLAY_FIXES.items():
        text = text.replace(wrong, right)
    return text


crops_found = list_subdirs(dataset_root)
unexpected = set(crops_found) - EXPECTED_CROPS
if unexpected:
    print(f"⚠️  Pastas de cultura inesperadas: {sorted(unexpected)}")

classes: list[dict] = []
for crop in crops_found:
    for disease_folder in list_subdirs(os.path.join(dataset_root, crop)):
        disease = display_name(disease_folder)
        classes.append({
            "index": len(classes),
            "name": f"{crop} - {disease}",
            "crop": crop,
            "crop_pt": CROP_PT.get(crop, crop.lower()),
            "disease": disease,
            "folder": f"{crop}/{disease_folder}",
        })

class_names = [c["name"] for c in classes]
NUM_CLASSES = len(classes)
print(f"Encontradas {NUM_CLASSES} classes:")
for c in classes:
    print(f"  {c['index']:2d}  {c['name']}")
if NUM_CLASSES != EXPECTED_NUM_CLASSES:
    print(f"\n⚠️  Esperava {EXPECTED_NUM_CLASSES} classes. Confirma a árvore de pastas acima.")

In [ ]:
def cache_image(src: str, dst: str) -> str | None:
    """Abre, corrige a orientação, reduz e guarda a imagem. Devolve o erro, ou None se correr bem."""
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return None  # já feito numa execução anterior desta sessão
    try:
        with Image.open(src) as img:
            img.draft("RGB", (RESIZE_SIZE, RESIZE_SIZE))  # JPEG: descomprime já em tamanho reduzido (rápido)
            img = ImageOps.exif_transpose(img)
            img = img.convert("RGB")
            width, height = img.size
            scale = RESIZE_SIZE / min(width, height)
            if scale < 1:
                img = img.resize((round(width * scale), round(height * scale)), Image.BICUBIC)
            img.save(dst, "JPEG", quality=95)
        return None
    except Exception as error:
        if os.path.exists(dst):
            os.remove(dst)
        return f"{type(error).__name__}: {error}"


jobs = []          # (origem, destino, índice da classe)
ignored_files = []  # ficheiros que não são imagens
for c in classes:
    class_dir = os.path.join(dataset_root, c["folder"])
    out_dir = os.path.join(CACHE_DIR, f"{c['index']:02d}")
    os.makedirs(out_dir, exist_ok=True)
    files = []
    for dirpath, _, names in os.walk(class_dir):
        for name in names:
            path = os.path.join(dirpath, name)
            (files if is_image_file(name) else ignored_files).append(path)
    for i, src in enumerate(sorted(files)):
        jobs.append((src, os.path.join(out_dir, f"{i:05d}.jpg"), c["index"]))

print(f"{len(jobs)} imagens para verificar; {len(ignored_files)} ficheiros ignorados por não serem imagens.")

with ThreadPoolExecutor(max_workers=8) as pool:
    errors = list(tqdm(pool.map(lambda job: cache_image(job[0], job[1]), jobs),
                       total=len(jobs), desc="A verificar e reduzir"))

records, corrupted = [], []
for (src, dst, label), error in zip(jobs, errors):
    if error is None:
        records.append({"path": dst, "label": label, "source": src})
    else:
        corrupted.append({"source": src, "label": label, "erro": error})

df = pd.DataFrame(records)
df["class_name"] = df["label"].map(lambda i: class_names[i])
df["crop"] = df["label"].map(lambda i: classes[i]["crop"])

print(f"\n✅ Imagens válidas: {len(df)}")
print(f"❌ Imagens corrompidas (excluídas): {len(corrupted)}")
for item in corrupted[:10]:
    print(f"   {os.path.relpath(item['source'], dataset_root)} → {item['erro']}")
if ignored_files[:5]:
    print("Exemplos de ficheiros ignorados:", [os.path.basename(p) for p in ignored_files[:5]])

In [ ]:
corrupted_per_class = Counter(item["label"] for item in corrupted)
counts_table = pd.DataFrame({
    "classe": class_names,
    "cultura": [c["crop"] for c in classes],
    "imagens": [int((df["label"] == i).sum()) for i in range(NUM_CLASSES)],
    "corrompidas": [corrupted_per_class.get(i, 0) for i in range(NUM_CLASSES)],
})
counts_table["% do total"] = (100 * counts_table["imagens"] / counts_table["imagens"].sum()).round(1)
display(counts_table)

print("Imagens por cultura:")
print(counts_table.groupby("cultura")["imagens"].sum().to_string())

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(counts_table["classe"], counts_table["imagens"], color="seagreen")
ax.invert_yaxis()
ax.set_xlabel("Número de imagens")
ax.set_title("Imagens por classe — repara no desequilíbrio")
plt.tight_layout()
plt.show()

In [ ]:
# Uma imagem de exemplo de cada classe
rng = random.Random(SEED)
cols = 6
rows = int(np.ceil(NUM_CLASSES / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.9))
for ax in axes.flat:
    ax.axis("off")
for i, ax in enumerate(axes.flat[:NUM_CLASSES]):
    example = rng.choice(df.loc[df["label"] == i, "path"].tolist())
    ax.imshow(Image.open(example))
    ax.set_title(class_names[i].replace(" - ", "\n"), fontsize=9)
plt.tight_layout()
plt.show()

## 4. Divisão treino / validação / teste (70 / 15 / 15)

Cada conjunto tem um papel diferente:
- **Treino (70%)** — as imagens com que o modelo aprende.
- **Validação (15%)** — usada durante o treino para escolher a melhor época e decidir quando parar. O modelo não aprende com elas, mas nós tomamos decisões com base nelas.
- **Teste (15%)** — guardado "na gaveta" até ao fim. É a única medida honesta de como o modelo se vai portar com fotos novas.

**Estratificada** quer dizer que cada classe aparece com a mesma proporção nos três conjuntos. Sem isso, uma classe rara podia ficar quase sem imagens no teste, por azar.

A **seed fixa** (42) garante que, se correres o notebook outra vez — por exemplo, depois de o Colab desligar — a divisão é exactamente a mesma. Isto é essencial para retomar o treino: se a divisão mudasse, imagens de teste podiam passar para o treino.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

for name, part in [("Treino", train_df), ("Validação", val_df), ("Teste", test_df)]:
    print(f"{name:10s}: {len(part):6d} imagens ({100 * len(part) / len(df):.1f}%)")

split_table = pd.DataFrame({
    "classe": class_names,
    "treino": train_df["label"].value_counts().sort_index().values,
    "validação": val_df["label"].value_counts().sort_index().values,
    "teste": test_df["label"].value_counts().sort_index().values,
})
split_table["% treino"] = (100 * split_table["treino"]
                           / split_table[["treino", "validação", "teste"]].sum(axis=1)).round(1)
display(split_table)

## 5. Data augmentation (só no treino)

**Data augmentation** é criar, em cada época, versões ligeiramente diferentes de cada foto de treino: rodada, virada, com outro brilho, recortada noutro sítio. O modelo nunca vê a mesma imagem duas vezes exactamente igual, por isso é obrigado a aprender **o que é a doença** e não pormenores de uma foto concreta. Também imita a realidade: os agricultores vão tirar fotos com luz, ângulos e distâncias muito diferentes.

As transformações usadas no **treino**:
- **Rotação** até ±20° e **flips** horizontal e vertical — uma folha doente continua doente de pernas para o ar;
- **Brilho, contraste, saturação e tom** — sol forte, sombra, câmaras diferentes;
- **Recorte aleatório** (60–100% da imagem) redimensionado para 224 px — a folha mais perto ou mais longe.

Na **validação e no teste não há augmentation**: só redimensionamos (lado menor 256) e recortamos o centro (224). Queremos medir o modelo em imagens "normais", sempre da mesma forma.

No fim, os valores de cada canal de cor são **normalizados** com a média e o desvio-padrão do ImageNet, porque foi assim que o modelo foi pré-treinado.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CropImageDataset(Dataset):
    """Lê as imagens da cache e aplica as transformações."""

    def __init__(self, frame: pd.DataFrame, transform: transforms.Compose) -> None:
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].tolist()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        with Image.open(self.paths[index]) as img:
            image = img.convert("RGB")
        return self.transform(image), self.labels[index]


def make_loader(frame: pd.DataFrame, transform: transforms.Compose, shuffle: bool) -> DataLoader:
    """Cria um DataLoader, que entrega as imagens à GPU em lotes (batches)."""
    return DataLoader(
        CropImageDataset(frame, transform),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
    )


train_loader = make_loader(train_df, train_transform, shuffle=True)
val_loader = make_loader(val_df, eval_transform, shuffle=False)
test_loader = make_loader(test_df, eval_transform, shuffle=False)
print(f"Lotes por época — treino: {len(train_loader)}, validação: {len(val_loader)}, teste: {len(test_loader)}")

In [ ]:
def denormalize(tensor: torch.Tensor) -> np.ndarray:
    """Desfaz a normalização para conseguirmos ver a imagem."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


# A mesma foto de treino, 8 vezes, com augmentations diferentes
example_row = train_df.sample(1, random_state=SEED).iloc[0]
with Image.open(example_row["path"]) as img:
    original = img.convert("RGB")

fig, axes = plt.subplots(1, 9, figsize=(20, 2.8))
axes[0].imshow(original)
axes[0].set_title("original", fontsize=9)
for ax in axes[1:]:
    ax.imshow(denormalize(train_transform(original)))
for ax in axes:
    ax.axis("off")
fig.suptitle(example_row["class_name"])
plt.tight_layout()
plt.show()

## 6. Modelo: EfficientNet-B0 pré-treinado + pesos por classe

**Transfer learning / fine-tuning.** Não treinamos um modelo do zero (precisaríamos de milhões de imagens). Partimos do **EfficientNet-B0 pré-treinado no ImageNet**: um modelo que já aprendeu, em 1,2 milhões de fotos, a reconhecer formas, texturas, manchas e cores. Trocamos só a última camada (que no ImageNet escolhia entre 1000 objectos) por uma nova com **22 saídas**, uma por classe, e depois ajustamos o modelo inteiro às nossas fotos. O EfficientNet-B0 é pequeno (~4 milhões de parâmetros, ~16 MB) e rápido — bom para um servidor modesto.

**Desequilíbrio entre classes.** Algumas classes têm muito mais fotos do que outras. Sem cuidado, o modelo aprende a "apostar" nas classes grandes, porque assim acerta mais vezes em média, e ignora as pequenas. A solução é dar **pesos por classe** na função de perda (*loss*):

$$\text{peso}_c = \frac{N}{K \cdot n_c}$$

onde $N$ é o total de imagens de treino, $K$ o número de classes e $n_c$ as imagens da classe $c$. Uma classe com metade das fotos da média tem peso 2: errar nela "custa" o dobro, e o modelo é obrigado a prestar-lhe atenção.

**Optimizador e learning rate.** Usamos **AdamW** com um learning rate pequeno (3e-4), para não estragar o que o modelo já sabia, e um **scheduler cosine**, que vai baixando o learning rate ao longo do treino para afinar no fim. Usamos ainda **mixed precision** (cálculos em 16 bits onde é seguro), que torna o treino quase 2× mais rápido na GPU T4.

In [ ]:
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=NUM_CLASSES).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"EfficientNet-B0 com {NUM_CLASSES} saídas: {n_params / 1e6:.2f} M parâmetros "
      f"(~{n_params * 4 / 1024**2:.0f} MB em float32)")

train_counts = np.bincount(train_df["label"], minlength=NUM_CLASSES)
class_weights = len(train_df) / (NUM_CLASSES * train_counts)
weights_table = pd.DataFrame({"classe": class_names, "imagens de treino": train_counts,
                              "peso na loss": class_weights.round(2)})
display(weights_table.sort_values("peso na loss", ascending=False))

criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

## 7. Treino, com checkpoint para retomar

Em cada **época** o modelo vê todas as imagens de treino uma vez e é avaliado na validação. Mostramos:
- **loss** (quanto o modelo erra — quanto mais baixo, melhor) e **accuracy** no treino e na validação;
- **F1 macro** na validação: a média do F1 de todas as classes, em que as classes pequenas contam tanto como as grandes.

**Melhor época e early stopping.** Sempre que a loss de validação melhora, guardamos o modelo. Se passar `PATIENCE` (4) épocas sem melhorar, paramos: o modelo começou a **decorar** o treino (*overfitting*) em vez de aprender.

**Checkpoint no Drive (`best.pt`, ~16 MB).** O Colab gratuito pode desligar a qualquer momento. Para não perdermos tudo:
- o melhor modelo é guardado primeiro em `/content` (rápido, sem limite);
- é copiado para o Drive **só quando melhora e já passaram pelo menos 3 épocas desde a última cópia**, e uma última vez no fim do treino — poucas escritas, para não encher o Lixo do Drive;
- o checkpoint leva **só o necessário**: os pesos, a época, a melhor loss, o estado do scheduler e o histórico. **Não** leva o estado interno do AdamW (seriam mais ~32 MB). Ao retomar, o AdamW reconstrói esse estado em poucas centenas de passos — a diferença no resultado é mínima.

**Se o Colab desligar:** volta a abrir o notebook e corre tudo de novo, desde o início. Os dados são preparados outra vez (a divisão sai igual, por causa da seed) e esta célula encontra o `best.pt` no Drive e **continua a partir da melhor época guardada**.

In [ ]:
def train_one_epoch() -> tuple[float, float]:
    """Uma passagem completa pelo treino. Devolve (loss média, accuracy)."""
    model.train()
    total_loss, correct, seen = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc="treino", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        seen += labels.size(0)
    return total_loss / seen, correct / seen


@torch.no_grad()
def predict(loader: DataLoader) -> tuple[float, np.ndarray, np.ndarray]:
    """Avalia o modelo num conjunto. Devolve (loss média, classes reais, classes previstas)."""
    model.eval()
    total_loss, seen = 0.0, 0
    y_true, y_pred = [], []
    for images, labels in tqdm(loader, desc="avaliação", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        seen += labels.size(0)
        y_true.append(labels.cpu().numpy())
        y_pred.append(logits.argmax(dim=1).cpu().numpy())
    return total_loss / seen, np.concatenate(y_true), np.concatenate(y_pred)


def save_checkpoint(path: str, epoch: int, best_val_loss: float, history: list[dict]) -> None:
    """Guarda só o necessário para retomar (sem o estado do AdamW)."""
    torch.save({
        "model_state": {k: v.cpu() for k, v in model.state_dict().items()},
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "scheduler_state": scheduler.state_dict(),
        "history": history,
        "class_names": class_names,
        "seed": SEED,
    }, path)


def copy_to_drive(local_path: str, drive_path: str) -> bool:
    """Copia um ficheiro para o Drive, se houver espaço. Não interrompe o treino se falhar."""
    size_mb = os.path.getsize(local_path) / 1024**2
    needed = 5 if os.path.exists(drive_path) else size_mb + 5  # substituir quase não ocupa espaço extra
    if drive_free_mb() < needed:
        print(f"   ⚠️  Sem espaço no Drive ({drive_free_mb():.0f} MB livres). Checkpoint NÃO copiado.")
        return False
    try:
        shutil.copyfile(local_path, drive_path)
        return True
    except OSError as error:
        print(f"   ⚠️  Falhou a cópia para o Drive: {error}")
        return False

In [ ]:
history: list[dict] = []
start_epoch = 1
best_val_loss = float("inf")
best_epoch = None
last_drive_epoch = None

if os.path.exists(DRIVE_CKPT_PATH):
    checkpoint = torch.load(DRIVE_CKPT_PATH, map_location=device, weights_only=True)
    if checkpoint["class_names"] != class_names:
        raise RuntimeError("O best.pt no Drive foi treinado com outras classes. Apaga-o ou muda o caminho.")
    model.load_state_dict(checkpoint["model_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    # O scheduler cosine calcula o próximo learning rate a partir do actual,
    # por isso repomos no optimizador o learning rate em que tínhamos ficado.
    for group, lr in zip(optimizer.param_groups, scheduler.get_last_lr()):
        group["lr"] = lr
    history = checkpoint["history"]
    best_val_loss = checkpoint["best_val_loss"]
    best_epoch = last_drive_epoch = checkpoint["epoch"]
    start_epoch = checkpoint["epoch"] + 1
    shutil.copyfile(DRIVE_CKPT_PATH, BEST_LOCAL_PATH)
    print(f"▶️  A retomar a partir da época {checkpoint['epoch']} (melhor loss de validação: {best_val_loss:.4f})")
else:
    print("▶️  Sem checkpoint no Drive: treino desde o início.")

epochs_without_improvement = 0
for epoch in range(start_epoch, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch()
    val_loss, y_val, y_val_pred = predict(val_loader)
    # float(): números simples do Python, para o checkpoint carregar com weights_only=True
    val_acc = float(accuracy_score(y_val, y_val_pred))
    val_f1 = float(f1_score(y_val, y_val_pred, average="macro"))
    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                    "val_loss": val_loss, "val_acc": val_acc, "val_f1_macro": val_f1, "lr": current_lr})
    print(f"Época {epoch:2d}/{EPOCHS} | treino: loss {train_loss:.4f}, acc {train_acc:.3f} | "
          f"validação: loss {val_loss:.4f}, acc {val_acc:.3f}, F1 {val_f1:.3f} | "
          f"{(time.time() - start) / 60:.1f} min")

    if val_loss < best_val_loss:
        best_val_loss, best_epoch = val_loss, epoch
        epochs_without_improvement = 0
        save_checkpoint(BEST_LOCAL_PATH, epoch, best_val_loss, history)
        if last_drive_epoch is None or epoch - last_drive_epoch >= MIN_EPOCHS_BETWEEN_DRIVE_SAVES:
            if copy_to_drive(BEST_LOCAL_PATH, DRIVE_CKPT_PATH):
                last_drive_epoch = epoch
                print(f"   💾 Melhor modelo até agora — guardado no Drive (livre: {drive_free_mb():.0f} MB)")
        else:
            print("   💾 Melhor modelo até agora — guardado localmente (vai para o Drive mais tarde)")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"⏹️  Early stopping: {PATIENCE} épocas sem melhorar a validação.")
            break

# Cópia final: garante que o Drive tem mesmo o melhor modelo
if best_epoch is not None and best_epoch != last_drive_epoch and os.path.exists(BEST_LOCAL_PATH):
    if copy_to_drive(BEST_LOCAL_PATH, DRIVE_CKPT_PATH):
        last_drive_epoch = best_epoch
        print("💾 Cópia final do melhor modelo para o Drive.")

print(f"\n✅ Treino terminado. Melhor época: {best_epoch} (loss de validação {best_val_loss:.4f})")

In [ ]:
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist["epoch"], hist["train_loss"], "o-", label="treino")
axes[0].plot(hist["epoch"], hist["val_loss"], "o-", label="validação")
axes[0].set_title("Loss (mais baixo é melhor)")
axes[1].plot(hist["epoch"], hist["train_acc"], "o-", label="treino")
axes[1].plot(hist["epoch"], hist["val_acc"], "o-", label="validação")
axes[1].plot(hist["epoch"], hist["val_f1_macro"], "s--", label="validação (F1 macro)")
axes[1].set_title("Accuracy / F1 (mais alto é melhor)")
for ax in axes:
    ax.set_xlabel("época")
    ax.axvline(best_epoch, color="grey", linestyle=":", label="melhor época")
    ax.legend()
plt.tight_layout()
plt.show()

**Como ler estas curvas:** se a loss de treino continua a descer mas a de validação sobe, o modelo está a **decorar** o treino (overfitting) — é exactamente quando o early stopping actua. Se as duas descem juntas até ao fim, o modelo talvez beneficiasse de mais épocas.

## 8. Avaliação no conjunto de TESTE

Agora, e **só agora**, usamos o conjunto de teste, com o **melhor** modelo (o da melhor época, não o da última). Estas são as métricas que vão para o README do projecto.

- **Accuracy:** percentagem de fotos bem classificadas.
- **Precision** de uma classe: das vezes que o modelo disse "é esta doença", quantas estavam certas? (Baixa → alarmes falsos.)
- **Recall** de uma classe: das fotos que eram mesmo desta doença, quantas o modelo apanhou? (Baixo → doenças que passam despercebidas — o mais grave para o agricultor.)
- **F1:** um equilíbrio entre precision e recall.
- **Matriz de confusão:** cada linha é a classe verdadeira, cada coluna a classe prevista. A diagonal são os acertos; fora da diagonal vemos **que doenças o modelo confunde entre si**.

In [ ]:
def load_best_weights() -> None:
    """Carrega os pesos da melhor época (local, ou do Drive se a sessão foi reiniciada)."""
    final_drive_path = os.path.join(DRIVE_MODELS_DIR, FINAL_MODEL_NAME)
    if os.path.exists(BEST_LOCAL_PATH):
        state = torch.load(BEST_LOCAL_PATH, map_location=device, weights_only=True)["model_state"]
    elif os.path.exists(DRIVE_CKPT_PATH):
        state = torch.load(DRIVE_CKPT_PATH, map_location=device, weights_only=True)["model_state"]
    elif os.path.exists(final_drive_path):
        state = torch.load(final_drive_path, map_location=device, weights_only=True)
    else:
        raise FileNotFoundError("Não encontrei nenhum modelo treinado. Corre o treino primeiro.")
    model.load_state_dict(state)


load_best_weights()
test_loss, y_test, y_test_pred = predict(test_loader)
test_acc = accuracy_score(y_test, y_test_pred)
test_f1_macro = f1_score(y_test, y_test_pred, average="macro")
test_f1_weighted = f1_score(y_test, y_test_pred, average="weighted")

print(f"Accuracy no teste:  {test_acc:.4f}  ({(y_test == y_test_pred).sum()} de {len(y_test)} fotos)")
print(f"F1 macro no teste:  {test_f1_macro:.4f}")
print(f"F1 ponderado:       {test_f1_weighted:.4f}\n")

labels_range = list(range(NUM_CLASSES))
print(classification_report(y_test, y_test_pred, labels=labels_range,
                            target_names=class_names, digits=3, zero_division=0))
report_dict = classification_report(y_test, y_test_pred, labels=labels_range,
                                    target_names=class_names, output_dict=True, zero_division=0)

In [ ]:
cm = confusion_matrix(y_test, y_test_pred, labels=labels_range)
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)  # cada linha soma 1 (recall na diagonal)

fig, ax = plt.subplots(figsize=(15, 13))
im = ax.imshow(cm_norm, cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(labels_range)
ax.set_yticks(labels_range)
ax.set_xticklabels(class_names, rotation=90, fontsize=9)
ax.set_yticklabels(class_names, fontsize=9)
ax.set_xlabel("Classe prevista pelo modelo")
ax.set_ylabel("Classe verdadeira")
ax.set_title(f"Matriz de confusão (teste, normalizada por linha) — accuracy {test_acc:.3f}")
for i in labels_range:
    for j in labels_range:
        if cm_norm[i, j] >= 0.01:
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
CM_LOCAL_PATH = f"{WORK_DIR}/confusion_matrix.png"
fig.savefig(CM_LOCAL_PATH, dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# Accuracy por cultura e as confusões mais frequentes
test_results = pd.DataFrame({"crop": [classes[i]["crop"] for i in y_test], "correct": y_test == y_test_pred})
crop_accuracy = test_results.groupby("crop")["correct"].mean().round(4).to_dict()
print("Accuracy por cultura no teste:")
for crop, acc in crop_accuracy.items():
    print(f"  {crop:8s} ({CROP_PT.get(crop, '')}): {acc:.3f}")

confusions = [(cm[i, j], class_names[i], class_names[j])
              for i in labels_range for j in labels_range if i != j and cm[i, j] > 0]
confusions.sort(reverse=True)
print("\nConfusões mais frequentes (verdadeira → prevista):")
for count, true_name, pred_name in confusions[:10]:
    print(f"  {count:3d}×  {true_name}  →  {pred_name}")

## 9. Exportar para `MyDrive/machamba-ia/models/`

Ficheiros gerados:

| Ficheiro | O que é |
|---|---|
| `machamba_efficientnet_b0.pt` | **Só os pesos** do melhor modelo (~16 MB). É o que a API vai carregar na Fase 2. |
| `labels.json` | Índice → classe, cultura, doença, e o pré-processamento a usar nas fotos novas. |
| `metrics.json` | Métricas do teste, histórico do treino e configuração (para o README). |
| `confusion_matrix.png` | O gráfico da matriz de confusão. |

**O modelo final substitui o `best.pt`**, para não ocupar espaço a dobrar: escrevemos os pesos finais **por cima** do `best.pt` e depois só lhe mudamos o nome. Assim o ficheiro não é apagado e não vai uma cópia extra para o Lixo.

⚠️ Depois disto o `best.pt` deixa de existir. Se voltares a correr a secção 7, o treino recomeça do zero.

In [ ]:
final_local_path = f"{WORK_DIR}/{FINAL_MODEL_NAME}"
final_drive_path = os.path.join(DRIVE_MODELS_DIR, FINAL_MODEL_NAME)

# 1) Modelo final: só os pesos, em CPU (assim carrega em qualquer máquina, com ou sem GPU)
load_best_weights()
torch.save({k: v.cpu() for k, v in model.state_dict().items()}, final_local_path)
final_mb = os.path.getsize(final_local_path) / 1024**2
print(f"Modelo final: {final_mb:.1f} MB")

if os.path.exists(DRIVE_CKPT_PATH):
    shutil.copyfile(final_local_path, DRIVE_CKPT_PATH)  # escreve por cima do best.pt...
    os.replace(DRIVE_CKPT_PATH, final_drive_path)       # ...e muda-lhe o nome
    print(f"✅ best.pt substituído por {FINAL_MODEL_NAME}")
elif copy_to_drive(final_local_path, final_drive_path):
    print(f"✅ {FINAL_MODEL_NAME} copiado para o Drive")
else:
    print(f"❌ Não consegui guardar o modelo no Drive. Descarrega-o à mão: {final_local_path}")

# 2) labels.json
labels_info = {
    "model": "efficientnet_b0",
    "num_classes": NUM_CLASSES,
    "image_size": IMAGE_SIZE,
    "resize_size": RESIZE_SIZE,
    "mean": list(IMAGENET_MEAN),
    "std": list(IMAGENET_STD),
    "preprocessing": "corrigir orientação EXIF → RGB → lado menor 256 → recorte central 224 → normalizar",
    "classes": {
        str(c["index"]): {"class": c["name"], "crop": c["crop"], "crop_pt": c["crop_pt"],
                          "disease": c["disease"], "folder": c["folder"]}
        for c in classes
    },
}

# 3) metrics.json
metrics = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model": "efficientnet_b0 (timm, pré-treinado no ImageNet)",
    "dataset": "CCMT Raw Data (Crop Pest and Disease Detection, Gana)",
    "test": {
        "accuracy": round(test_acc, 4),
        "f1_macro": round(test_f1_macro, 4),
        "f1_weighted": round(test_f1_weighted, 4),
        "loss": round(test_loss, 4),
        "accuracy_per_crop": crop_accuracy,
        "per_class": {name: {k: round(v, 4) for k, v in values.items()}
                      for name, values in report_dict.items() if name in class_names},
        "confusion_matrix": cm.tolist(),
    },
    "data": {
        "images_valid": len(df),
        "images_corrupted": len(corrupted),
        "files_ignored": len(ignored_files),
        "train": len(train_df), "val": len(val_df), "test": len(test_df),
    },
    "training": {
        "best_epoch": best_epoch,
        "best_val_loss": round(best_val_loss, 4),
        "history": history,
    },
    "config": {
        "seed": SEED, "image_size": IMAGE_SIZE, "resize_size": RESIZE_SIZE, "batch_size": BATCH_SIZE,
        "epochs_max": EPOCHS, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "patience": PATIENCE, "split": "70/15/15 estratificado", "class_weighted_loss": True,
        "torch": torch.__version__, "timm": timm.__version__,
    },
}

for name, content in [("labels.json", labels_info), ("metrics.json", metrics)]:
    local = f"{WORK_DIR}/{name}"
    with open(local, "w", encoding="utf-8") as f:
        json.dump(content, f, ensure_ascii=False, indent=2)
    copy_to_drive(local, os.path.join(DRIVE_MODELS_DIR, name))

# 4) confusion_matrix.png
copy_to_drive(CM_LOCAL_PATH, os.path.join(DRIVE_MODELS_DIR, "confusion_matrix.png"))

print(f"\nConteúdo de {DRIVE_MODELS_DIR}:")
for name in sorted(os.listdir(DRIVE_MODELS_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_MODELS_DIR, name)) / 1024**2
    print(f"  {name:32s} {size:7.2f} MB")
print(f"\nEspaço livre no Drive: {drive_free_mb():.0f} MB")

### Último passo: garantir que tudo foi gravado no Drive

O Colab escreve no Drive em segundo plano. A célula abaixo **força a gravação** de tudo o que está pendente e desliga o Drive. Corre-a antes de fechar o notebook.

Depois:
1. Em drive.google.com, abre `machamba-ia/models/` e descarrega os 4 ficheiros.
2. Coloca-os na pasta `models/` do projecto local (`C:\Projectos\machamba-ia\models\`).
3. **Esvazia o Lixo do Drive** para recuperares o espaço das versões antigas do `best.pt`.

In [ ]:
drive.flush_and_unmount()
print("✅ Drive gravado e desligado. Podes fechar o notebook.")